In [204]:
MANUSCRIPTS = ["P5881", "P400", "CCCP578", "P3465", "P3466", "A4095","P3471","R2536" ,"P3473", "P2789", "P3475" ,"Met-1981.373"]
CHAPTERS = ['Lv', 'Im', 'Oc', 'Mc', 'Kd', 'Km','Ag','Lj']

In [205]:
import json
import os
with open("manuscripts.json", "r", encoding="utf-8") as file:
    manuscripts_src = json.loads(file.read())

with open("units.json", "r", encoding="utf-8") as file:
    units_src = json.loads(file.read())
with open("segments.json", "r", encoding="utf-8") as file:
    segments_src = json.loads(file.read())
with open("pages.json", "r", encoding="utf-8") as file:
    pages_src = json.loads(file.read())
with open("text.json", "r", encoding="utf-8") as file:
    text_src = json.loads(file.read())
with open("lines.json", "r", encoding="utf-8") as file:
    lines_src = json.loads(file.read())
with open("images.json", "r", encoding="utf-8") as file:
    images_src = json.loads(file.read())

In [206]:
def get_mss_siglum_to_id(ms_list):
    res = {}
    for siglum in ms_list:
        for ms in manuscripts_src:
            if siglum == ms['siglum']:
                res[siglum] = ms['id']
    return res
def get_units(chapter):
    units = [x for x in units_src if x.get('frame', '') == chapter]
    unit_ids = {x['id'] for x in units}
    for unit in units:
        unit['longestSegment'] = {'count': 0, 'siglum': ''}
    return units, unit_ids
def format_units(units, root_unit_title):
    root_unit = next((x for x in units if x['title'] == root_unit_title), None)
    root_unit['formattedOrder'] = root_unit['frame']
    def get_childern(root_id, order):
        children = []
        for unit in units:
            if unit['parentId'] == root_id:
                unit['formattedOrder'] = f"{root_unit['frame']}.{unit['order']}"
                children.append(unit)
                children += get_childern(unit['id'], unit['order'])
        return children
    return root_unit, get_childern(root_unit['id'], f"{root_unit['order']}")
def get_segments(ms_id, unit_ids):
    segments = [x for x in segments_src if x['unitId'] in unit_ids and x['mediumId'] == ms_id]
    return {x['unitId']: x for x in segments}
def get_segment_pages(ms_id, segment):
    pages = [x for x in pages_src if x['mediumId'] == ms_id and x['number']>= segment['startPage'] and x['number'] <= segment['endPage']  ]
    return pages

def get_segment_bounding_pages(pages, segment):
    first = next((x for x in pages if x['number'] == segment['startPage']), None)
    last = next((x for x in pages if x['number'] == segment['endPage']), None)
    return first, last
def get_page_body_lines(page):
    element_ids = {x['id'] for x in text_src if x['pageId'] == page['id'] and 'main' in x['position']}
    lines = [x for x in lines_src if x['elementId'] in element_ids]
    for line in lines:
        line['pageNumber'] = page['number']
        line['pageId'] = page['id']
    lines.sort(key=lambda item: item['order'])
    return lines

def get_segment_lines(lines, segment, start_page_id, end_page_id):
    if segment["startPage"] == segment["endPage"]:
        return [
            x
            for x in lines
            if x["order"] >= segment["startLine"] and x["order"] <= segment["endLine"]
        ]
    if segment["endPage"] - segment["startPage"] == 1:
        lines_from_start_page = [
            x
            for x in lines
            if x["order"] >= segment["startLine"] and x["pageId"] == start_page_id
        ]
        lines_from_start_page.sort(key=lambda item: item["order"])
        lines_from_end_page = [
            x
            for x in lines
            if x["order"] <= segment["endLine"] and x["pageId"] == end_page_id
        ]
        lines_from_end_page.sort(key=lambda item: item["order"])
        return lines_from_start_page + lines_from_end_page
    if segment["endPage"] - segment["startPage"] > 1:
        lines_from_start_page = [
            x
            for x in lines
            if x["order"] >= segment["startLine"] and x["pageId"] == start_page_id
        ]
        lines_from_start_page.sort(key=lambda item: item["order"])
        lines_from_mid_pages = [
            x
            for x in lines
            if x["pageId"] != start_page_id and  x["pageId"] != end_page_id
        ]
        lines_from_mid_pages.sort(key=lambda item: (item["order"], item['pageNumber']))
        lines_from_end_page = [
            x
            for x in lines
            if x["order"] <= segment["endLine"] and x["pageId"] == end_page_id
        ]
        lines_from_end_page.sort(key=lambda item: item["order"])
        return lines_from_start_page + lines_from_mid_pages + lines_from_end_page

def get_line_data(line, start=None, end=None):
    if start is not None and end is not None:
        tokens = line['tokens'][start:end]
        states = line['states'][start:end]
    elif start is not None:
        tokens = line['tokens'][start:]
        states = line['states'][start:]
    elif end is not None:
        tokens = line['tokens'][:end]
        states = line['states'][:end]
    else:
        tokens = line['tokens']
        states = line['states']
    
    return {
        'tokens': tokens,
        'states': states,
        'lines': [line['order']] * len(tokens),
        'pages': [line['pageNumber']] * len(tokens),
        'breaks': [None if i != 0 or line['order'] != 0 else line['pageNumber'] 
                   for i in range(len(tokens))]
    }

def get_segment_data_same_line(lines, segment):
    line_data = get_line_data(lines[0], start=segment['startToken'], end=segment['endToken']+1)
    data = {
        'images': [],
        'lemmas': []
    }
    data.update(line_data)
    return data



def get_segment_data_same_page(lines, segment):
    data = {
        'tokens': [],
        'states': [],
        'lines': [],
        'pages': [],
        'images': [],
        'breaks': [],
        'lemmas': []
    }
    for line in lines:
        if line['order'] == segment['startLine']:
            line_data = get_line_data(line, start=segment['startToken'])
        elif line['order'] == segment['endLine']:
            line_data = get_line_data(line, end=segment['endToken']+1)
        else:
            line_data = get_line_data(line)
        
        data['tokens'] += line_data['tokens']
        data['states'] += line_data['states']
        data['lines'] += line_data['lines']
        data['pages'] += line_data['pages']
        data['breaks'] += line_data['breaks']

    return data


def get_segment_data_multiple_pages(lines, segment):
    data = {
        'tokens': [],
        'states': [],
        'lines': [],
        'pages': [],
        'images': [],
        'breaks': [],
        'lemmas': []
    }
    for line in lines:
        if line['order'] == segment['startLine'] and line['pageNumber'] == segment['startPage']:
            line_data = get_line_data(line, start=segment['startToken'])
        elif line['order'] == segment['endLine'] and line['pageNumber'] == segment['endPage']:
            line_data = get_line_data(line, end=segment['endToken']+1)
        else:
            line_data = get_line_data(line)
        
        data['tokens'] += line_data['tokens']
        data['states'] += line_data['states']
        data['lines'] += line_data['lines']
        data['pages'] += line_data['pages']
        data['breaks'] += line_data['breaks']

    return data


def get_segment_data(lines, segment):
    if segment['startPage'] == segment['endPage']:
        if segment['startLine'] == segment['endLine']:
            return get_segment_data_same_line(lines, segment)
        else:
            return get_segment_data_same_page(lines, segment)
    else:
        return get_segment_data_multiple_pages(lines, segment)


In [207]:
# chapter_units
for chapter in CHAPTERS:
  units, unit_ids = get_units(chapter)
  units.sort(key=lambda item: item["order"])
  units = [{'title': x['title'], 'number':x['order'] } for x in units]
  out_file = f"../apps/data-api/data/manuscripts/chapter_units/{chapter}.json"
  with open(out_file, "w", encoding='utf-8') as write_file:
    json.dump(units, write_file, indent=4)


In [208]:
# chapter_to_ms
mss_siglum_to_id = get_mss_siglum_to_id(MANUSCRIPTS)

for chapter in CHAPTERS:
  units, unit_ids = get_units(chapter)
  units.sort(key=lambda item: item["order"])
  units = [{'title': x['title'], 'number':x['order'] } for x in units]
  data =[]
  for i, ms in enumerate(MANUSCRIPTS):
    id = mss_siglum_to_id[ms]
    segments = get_segments(id, unit_ids).values()
    if len(segments) != 0:
      start_page = min([x['startPage'] for x in segments])
      end_page = max([x['endPage'] for x in segments])
      data.append({"manuscript": ms, 'from': start_page, 'to': end_page})

  out_file = f"../apps/data-api/data/manuscripts/chapter_to_ms/{chapter}.json"
  with open(out_file, "w", encoding='utf-8') as write_file:
    json.dump(data, write_file, indent=4)



In [209]:
print(lines_src[20])


{'elementId': '2ba2d549-7c6a-4994-bfab-d49080c398d2', 'id': '8af960af-bbb5-41d7-a7d6-2b1061f419e8', 'order': 3, 'region': [134, 875, 1724, 875, 1724, 1031, 134, 1031, 0], 'tokens': ['الشأن', 'فوُصف', 'للوزير', 'فوجّه', 'إليه', 'فأحضره', 'وناظره', 'على', 'أمر'], 'states': ['sound', 'sound', 'sound', 'sound', 'sound', 'sound', 'sound', 'sound', 'sound']}


In [210]:
# the data of each page in manuscripts
for ms_siglum in MANUSCRIPTS:
    for chapter in CHAPTERS:

        ms_id = mss_siglum_to_id[ms_siglum]
        units, unit_ids = get_units(chapter)
        segments = get_segments(ms_id, unit_ids)

        if segments:  # Check if segments is not empty
            for page_number in range(min([x['startPage'] for x in segments.values()]), max([x['endPage'] for x in segments.values()])+1):
                page_segments = [x for x in segments.values() if x['startPage'] <= page_number <= x['endPage']]
                page_units = [x for x in units if x['id'] in [y['unitId'] for y in page_segments]]
                page_data = {
                    "id": "",
                    "number": page_number,
                    "manuscript": ms_siglum,
                    "imageUrl": "", # Corrected the image url
                    "lines": [],
                    "unitPlaces": [],
                    "unitNames": []
                }

                # Check for units from other chapters
                for other_chapter in CHAPTERS:
                    if other_chapter != chapter:
                        other_units, other_unit_ids = get_units(other_chapter)
                        other_segments = get_segments(ms_id, other_unit_ids)
                        other_page_segments = [x for x in other_segments.values() if x['startPage'] <= page_number <= x['endPage']]
                        other_page_units = [x for x in other_units if x['id'] in [y['unitId'] for y in other_page_segments]]
                        for unit in other_page_units:
                            segment = next((x for x in other_page_segments if x['unitId'] == unit['id']), None)
                            if segment is not None:
                                page_data['unitPlaces'].append([segment['startLine'], segment['startToken']])
                                page_data['unitNames'].append([other_chapter + ' ' + unit['title'], unit['order']])

                for unit in page_units:
                    segment = next((x for x in page_segments if x['unitId'] == unit['id']), None)
                    if segment is not None:
                        page_data['unitPlaces'].append([segment['startLine'], segment['startToken']])
                        page_data['unitNames'].append([chapter + ' ' + unit['title'], unit['order']])
                for segment in page_segments:
                    pages = get_segment_pages(ms_id, segment)
                    for page in pages:
                        if page['number'] == page_number:
                            page_data['imageUrl']=page['image']
                            page_data['id']=page['id']
                            lines = get_page_body_lines(page)
                            for line in lines:
                                if line['tokens'] not in page_data['lines']: # Check if line already exists in page_data
                                    page_data['lines'].append(line['tokens'])
                               
                directory = f"../apps/data-api/data/manuscripts/test/{ms_siglum}/{chapter}"
                os.makedirs(directory, exist_ok=True)
                out_file = f"{directory}/{page_number}.json"  # Changed the json file name to match the page number
                with open(out_file, "w", encoding='utf-8') as write_file:
                    json.dump(page_data, write_file, indent=4, ensure_ascii=False)




In [ ]:
#all the pages in one manuscript
for ms_siglum in MANUSCRIPTS:
    all_pages = []
    for chapter in CHAPTERS:

        ms_id = mss_siglum_to_id[ms_siglum]
        units, unit_ids = get_units(chapter)
        segments = get_segments(ms_id, unit_ids)

        if segments:  # Check if segments is not empty
            for page_number in range(min([x['startPage'] for x in segments.values()]), max([x['endPage'] for x in segments.values()])+1):
                page_segments = [x for x in segments.values() if x['startPage'] <= page_number <= x['endPage']]
                page_units = [x for x in units if x['id'] in [y['unitId'] for y in page_segments]]
                page_data = {
                    "index": page_number,
                    "page_number": page_number,
                    "page_link": f"/manuscripts/{ms_siglum}/{chapter}/{page_number}"
                }
                # Check if page_number already exists in all_pages
                if page_number not in [page['page_number'] for page in all_pages]:
                    all_pages.append(page_data)

    # Sort all_pages by index before writing to file
    all_pages.sort(key=lambda page: page['index'])

    directory = f"../apps/data-api/data/manuscripts/test/{ms_siglum}"
    os.makedirs(directory, exist_ok=True)
    out_file = f"{directory}/allPages.json"
    with open(out_file, "w", encoding='utf-8') as write_file:
        json.dump(all_pages, write_file, indent=4, ensure_ascii=False)



In [ ]:
# the pages in one chapter 
for ms_siglum in MANUSCRIPTS:
    ms_id = mss_siglum_to_id[ms_siglum]
    for chapter in CHAPTERS:
        units, unit_ids = get_units(chapter)
        segments = get_segments(ms_id, unit_ids)
        pages_in_chapter = []

        if segments:  # Check if segments is not empty
            for page_number in range(min([x['startPage'] for x in segments.values()]), max([x['endPage'] for x in segments.values()])+1):
                page_segments = [x for x in segments.values() if x['startPage'] <= page_number <= x['endPage']]
                page_units = [x for x in units if x['id'] in [y['unitId'] for y in page_segments]]
                page_data = {
                    "index": page_number,
                    "page_number": page_number,
                    "page_link": f"/manuscripts/{ms_siglum}/{chapter}/{page_number}"
                }
                pages_in_chapter.append(page_data)

        directory = f"../apps/data-api/data/manuscripts/test/{ms_siglum}/{chapter}"
        os.makedirs(directory, exist_ok=True)
        out_file = f"{directory}/pagesInTheChapter.json"
        with open(out_file, "w", encoding='utf-8') as write_file:
            json.dump(pages_in_chapter, write_file, indent=4, ensure_ascii=False)


In [ ]:
for ms_siglum in MANUSCRIPTS:
    all_chapters = []
    last_page_number = 0
    for chapter in CHAPTERS:

        ms_id = mss_siglum_to_id[ms_siglum]
        units, unit_ids = get_units(chapter)
        segments = get_segments(ms_id, unit_ids)
        all_pages = []

        if segments:  # Check if segments is not empty
            for page_number in range(min([x['startPage'] for x in segments.values()]), max([x['endPage'] for x in segments.values()])+1):
                page_segments = [x for x in segments.values() if x['startPage'] <= page_number <= x['endPage']]
                page_units = [x for x in units if x['id'] in [y['unitId'] for y in page_segments]]
                page_data = {
                    "index": page_number,
                    "page_number": page_number,
                    "page_link": f"/manuscripts/{ms_siglum}/{chapter}/{page_number}"
                }
                all_pages.append(page_data)

        # Sort all_pages by page_number before adding to chapter_data
        all_pages.sort(key=lambda page: page['page_number'])

        chapter_data = {
            "chapter": chapter,
            "pages": all_pages
        }

        # Check if the first page number of the current chapter is smaller than the last page number of the last chapter
        if all_pages and all_pages[0]['page_number'] < last_page_number:
            # If so, insert the current chapter at the correct position
            for i, ch in enumerate(all_chapters):
                if ch['pages'] and all_pages[0]['page_number'] < ch['pages'][0]['page_number']:
                    all_chapters.insert(i, chapter_data)
                    break
        else:
            all_chapters.append(chapter_data)

        # Update the last page number
        if all_pages:
            last_page_number = all_pages[-1]['page_number']

    directory = f"../apps/data-api/data/manuscripts/test/{ms_siglum}"
    os.makedirs(directory, exist_ok=True)
    out_file = f"{directory}/allChapters.json"
    with open(out_file, "w", encoding='utf-8') as write_file:
        json.dump(all_chapters, write_file, indent=4, ensure_ascii=False)




In [ ]:
for ms_siglum in MANUSCRIPTS:
    gallery = []
    added_pages = set()  # Set to keep track of added pages
    for chapter in CHAPTERS:

        ms_id = mss_siglum_to_id[ms_siglum]
        units, unit_ids = get_units(chapter)
        segments = get_segments(ms_id, unit_ids)

        if segments:  # Check if segments is not empty
            for page_number in range(min([x['startPage'] for x in segments.values()]), max([x['endPage'] for x in segments.values()])+1):
                page_segments = [x for x in segments.values() if x['startPage'] <= page_number <= x['endPage']]
                page_units = [x for x in units if x['id'] in [y['unitId'] for y in page_segments]]
                page_data = {
                    "index": page_number,
                    "page_number": page_number,
                    "page_link": f"/manuscripts/{ms_siglum}/{chapter}/{page_number}"
                }
                for segment in page_segments:
                    pages = get_segment_pages(ms_id, segment)
                    for page in pages:
                        if page['number'] == page_number and page['id'] not in added_pages:  # Check if page has already been added
                            page_data['imageUrl']=page['image']
                            page_data['id']=page['id']
                            gallery.append({
                                "caption": f"{chapter} {page_number}",
                                "src": page_data['imageUrl'],
                                "thumb": page_data['imageUrl'],
                                "subHtml": f"{chapter} {page_number}",
                                "pageNumber": page_number  # Add page number for sorting
                            })
                            added_pages.add(page['id'])  # Add page id to the set of added pages

    # Sort the gallery by page_number before writing to file
    gallery.sort(key=lambda item: item['pageNumber'])

    directory = f"../apps/data-api/data/manuscripts/test/{ms_siglum}"
    os.makedirs(directory, exist_ok=True)
    out_file = f"{directory}/gallery.json"
    with open(out_file, "w", encoding='utf-8') as write_file:
        json.dump(gallery, write_file, indent=4, ensure_ascii=False)


